# LiteRT 온디바이스 이미지 분류 최적화: 최종 리포트

본 리포트는 **MobileNetV2** 모델을 모바일 기기에 배포하기 위해 수행한 **모델 변환, 양자화(Quantization), 그리고 성능 측정**의 전 과정을 상세히 기술합니다.

---

## 1. 모델 준비 (Model Source)

**모델 출처:** [PyTorch Torchvision](https://pytorch.org/vision/stable/models.html) 
**선정 이유:** MobileNetV2는 온디바이스 AI의 'Hello World'와 같은 표준 모델로, 경량화 효과를 검증하기에 가장 적합합니다.

우리는 `torchvision`에서 사전 학습된(Pretrained) 가중치를 가져와 시작합니다.

In [4]:
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd

# 소스 코드 패키지 경로 추가
sys.path.append(os.path.abspath(".."))

from src import run_comparison

# 그래프 스타일 설정
plt.style.use('ggplot')

## 2. 변환 및 양자화 방법론 (Conversion & Quantization Methods)

우리는 총 3가지 전략으로 모델을 변환했습니다. 각 전략은 **호환성**과 **성능** 사이의 트레이드오프를 가집니다.

### 전략 A: FP32 (Baseline)
- **도구:** `ai-edge-torch` (Google Official)
- **설명:** PyTorch 모델을 그대로 LiteRT(TFLite) 포맷으로 변환합니다. 가중치는 32비트 부동소수점(Float32)을 유지합니다.
- **특징:** 정확도는 원본과 동일하지만, 모델 크기가 크고(13MB) NPU 가속을 온전히 활용하기 어렵습니다.

### 전략 B: INT8 (Legacy ONNX Route)
- **도구:** `torch.onnx` -> `onnx2tf` -> `tflite`
- **설명:** PyTorch를 범용 포맷인 **ONNX**로 먼저 변환한 후, 이를 다시 TensorFlow 포맷으로 바꾸어 양자화합니다.
- **이유:** 초기 `ai-edge-torch`의 양자화 기능이 불안정하여 선택한 우회 경로입니다.
- **결과:** **3.8MB (72% 압축)** 달성 성공.

### 전략 C: INT8 (Modern ai-edge Hybrid)
- **도구:** `ai-edge-torch` (SavedModel Export) -> `TFLiteConverter`
- **설명:** **"SavedModel Intercept"** 기법을 사용했습니다.
    1. `ai-edge-torch`의 강력한 그래프 변환(Lowering) 기능을 사용해 **TensorFlow SavedModel**을 추출합니다.
    2. 버그가 있는 내부 양자화기 대신, 검증된 표준 `TFLiteConverter`를 사용하여 이 SavedModel을 양자화합니다.
- **의의:** 복잡한 ONNX 변환 없이도 구글의 최신 툴체인을 활용해 안정적인 INT8 모델을 얻을 수 있음을 입증했습니다.

## 3. 로컬 벤치마크 (Local Benchmark: x86 CPU)

개발 환경(PC)에서 모델 크기와 추론 속도를 측정합니다.

이 벤치마크는 **Interpreter API**와 새로운 **CompiledModel API** 두 가지 실행 방식의 성능을 비교 분석합니다.

> **주의:** x86 CPU는 FP32 연산에 최적화되어 있어, INT8 모델이 더 느리게 측정될 수 있습니다. 이는 정상적인 현상입니다.

In [5]:
# 로컬 PC에서 벤치마크 실행
local_results = run_comparison()
pd.DataFrame(local_results)

Searching for models in: /home/danya/workspace/study/lite-rt-study/models
Found 3 models. Starting benchmark...

Benchmarking /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2.tflite...
Benchmarking CompiledModel /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2.tflite...


INFO: [environment.cc:29] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:52] RegisterAccelerator: ptr=0x5f70b92a15d0, name=CpuAccelerator
INFO: [auto_registration.cc:168] CPU accelerator registered.
INFO: [compiled_model.cc:467] Flatbuffer model initialized directly from incoming litert model.


Benchmarking /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2_aiedge_int8.tflite...
Benchmarking CompiledModel /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2_aiedge_int8.tflite...


INFO: [accelerator_registry.cc:41] DestroyAccelerator: ptr=0x5f70b92a15d0, name=CpuAccelerator
INFO: [environment.cc:29] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:52] RegisterAccelerator: ptr=0x5f70b91fed10, name=CpuAccelerator
INFO: [auto_registration.cc:168] CPU accelerator registered.
INFO: [compiled_model.cc:467] Flatbuffer model initialized directly from incoming litert model.


Benchmarking /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2_legacy_int8.tflite...
Benchmarking CompiledModel /home/danya/workspace/study/lite-rt-study/models/mobilenet_v2_legacy_int8.tflite...


INFO: [accelerator_registry.cc:41] DestroyAccelerator: ptr=0x5f70b91fed10, name=CpuAccelerator
INFO: [environment.cc:29] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:52] RegisterAccelerator: ptr=0x5f70b4d2ffd0, name=CpuAccelerator
INFO: [auto_registration.cc:168] CPU accelerator registered.
INFO: [compiled_model.cc:467] Flatbuffer model initialized directly from incoming litert model.



COMPARISON RESULTS
+-----------------------------------------------+-------------+--------------------+------------+------------+-----------------+
| Model                                         |   Size (MB) |   Avg Latency (ms) |   Min (ms) |   Max (ms) |   Peak Mem (MB) |
+===============================================+=============+====================+============+============+=================+
| mobilenet_v2.tflite (Interpreter)             |       13.43 |               6.38 |       5.44 |       8.88 |         1280.37 |
+-----------------------------------------------+-------------+--------------------+------------+------------+-----------------+
| mobilenet_v2.tflite (Compiled)                |       13.43 |               5.87 |       5.64 |       6.24 |         1280.93 |
+-----------------------------------------------+-------------+--------------------+------------+------------+-----------------+
| mobilenet_v2_aiedge_int8.tflite (Interpreter) |        3.83 |              

INFO: [accelerator_registry.cc:41] DestroyAccelerator: ptr=0x5f70b4d2ffd0, name=CpuAccelerator


,file,size_mb,avg_ms,min_ms,max_ms,std_ms,mem_peak_mb
0,mobilenet_v2.tflite (Interpreter),13.426849,6.378012,5.442858,8.880615,1.155349,1280.371094
1,mobilenet_v2.tflite (Compiled),13.426849,5.870643,5.644083,6.237030,0.151997,1280.933594
2,mobilenet_v2_aiedge_int8.tflite (Interpreter),3.828400,3.602901,3.511190,3.774643,0.056807,1280.933594
3,mobilenet_v2_aiedge_int8.tflite (Compiled),3.828400,4.245815,3.504753,5.893230,0.909388,1280.933594
4,mobilenet_v2_legacy_int8.tflite (Interpreter),3.822128,3.500948,3.438473,3.583431,0.034818,1280.933594
5,mobilenet_v2_legacy_int8.tflite (Compiled),3.822128,3.570890,3.421068,3.859043,0.100408,1280.933594


## 4. 모바일 벤치마크 (Mobile Benchmark: Android ARM64)

실제 온디바이스 성능을 확인하기 위해, **ADB(Android Debug Bridge)**를 통해 연결된 안드로이드 기기에서 벤치마크를 수행합니다.

### 측정 방법
1. **도구:** 구글 공식 `benchmark_model` 바이너리 (Android arm64용)를 사용합니다.
2. **절차:**
    - 모델과 벤치마크 툴을 기기의 `/data/local/tmp/` 경로로 전송합니다.
    - `adb shell`을 통해 바이너리를 실행하고, **CPU, GPU, NPU** 각각의 Latency를 측정합니다.
3. **스크립트:** `src/benchmark_adb.py`가 이 과정을 자동화합니다.

In [6]:
# 안드로이드 기기가 연결되어 있다면 벤치마크 스크립트 실행
!python3 ../src/benchmark_adb.py

Setting up ADB environment...


/home/danya/.local/share/uv/python/cpython-3.10.19-linux-x86_64-gnu/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Pushing benchmark binary: /home/danya/workspace/study/lite-rt-study/bin/benchmark_model_android_arm64
Pushing model: mobilenet_v2.tflite
Pushing model: mobilenet_v2_aiedge_int8.tflite
Pushing model: mobilenet_v2_legacy_int8.tflite
Running benchmark for mobilenet_v2.tflite (GPU=False, NNAPI=False, AccelService=False)...
Running benchmark for mobilenet_v2.tflite (GPU=True, NNAPI=False, AccelService=False)...
Running benchmark for mobilenet_v2.tflite (GPU=False, NNAPI=False, AccelService=True)...
Running benchmark for mobilenet_v2_aiedge_int8.tflite (GPU=False, NNAPI=False, AccelService=False)...
Running benchmark for mobilenet_v2_aiedge_int8.tflite (GPU=True, NNAPI=False, AccelService=False)...
Running benchmark for mobilenet_v2_aiedge_int8.tflite (GPU=False, NNAPI=False, AccelService=True)...
Running benchmark for mobilenet_v2_legacy_int8.tflite (GPU=False, NNAPI=False, AccelService=False)...
Running benchmark for mobilenet_v2_legacy_int8.tflite (GPU=True, NNAPI=False, AccelService=Fals

## 5. 최종 결론 (Conclusion)

### 1. 양자화 효과 (Quantization Benefit)
- **용량:** 13.4MB → **3.8MB** (약 **72% 감소**)
- **속도 (Mobile CPU):** 68ms → **39ms** (약 **1.7배 가속**)

### 2. 하이브리드 전략의 승리
- 우리가 개발한 **"ai-edge Hybrid" (SavedModel Intercept)** 방식은 복잡한 ONNX 변환 없이도 **ONNX 방식과 동일한 고성능 INT8 모델**을 만들어냈습니다.
- 이는 향후 유지보수성과 구글 생태계 호환성 면에서 가장 권장되는 방식입니다.

### 3. 배포 가이드
- **모바일 앱:** INT8 모델 + NPU/CPU 가속 사용 권장.
- **PC/서버:** 저장 공간이 넉넉하다면 FP32 모델 사용 권장.